# Auto Loader

Auto Loader is Databricks's solution for **incrementally ingesting files from cloud storage** as they arrive. It is the recommended approach for Bronze layer ingestion at scale — handling everything from a few files per hour to millions of files continuously, with exactly-once guarantees.

In this notebook we'll cover:
- What Auto Loader does and how it tracks progress
- The `cloudFiles` source — syntax and key options
- Schema inference and the four schema evolution modes
- File discovery: directory listing vs file notification
- Checkpointing and trigger modes
- Auto Loader vs `COPY INTO` — which to use when

## The Problem Auto Loader Solves

Raw data lands in cloud storage continuously — sensor readings, application events, CDC exports. A naive ingestion approach would read the entire directory on every run and deduplicate against what's already loaded. That works for hundreds of files but fails at millions.

Auto Loader solves this with **stateful file tracking**:
- It remembers which files it has already processed (via a checkpoint)
- On each run, it processes **only new files** — no full directory scan, no deduplication needed
- It can discover new files either by listing the directory or by subscribing to cloud file-arrival events

Auto Loader is implemented as a **Structured Streaming source** using the `cloudFiles` format. This means it uses all of Spark Streaming's fault-tolerance and exactly-once guarantees automatically.

## Basic Syntax

```python
stream = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format",         "json")           # source file format
         .option("cloudFiles.schemaLocation",  "/ckpt/schema")  # schema inference store
         .load("s3://my-bucket/landing/events/")                # source path
)

stream.writeStream \
      .format("delta") \
      .outputMode("append") \
      .option("checkpointLocation", "/ckpt/events") \
      .trigger(availableNow=True) \
      .start("s3://my-bucket/bronze/events/")
```

The two required options:
- `cloudFiles.format` — the format of the source files (`json`, `csv`, `parquet`, `avro`, `text`, `binaryFile`)
- `cloudFiles.schemaLocation` — a path where Auto Loader stores the inferred schema across runs (can be the same directory as `checkpointLocation`)

The target `.start()` path (or `.table()` for a registered Delta table) is where the data is written.

## Schema Inference & Evolution

Auto Loader infers the schema from the first batch of files and stores it at `cloudFiles.schemaLocation`. On subsequent runs, it reuses the stored schema — no re-scanning needed.

### Key Schema Options

```python
.option("cloudFiles.inferColumnTypes", "true")    # infer actual types (default: all STRING)
.option("cloudFiles.schemaHints",                  # override specific columns
        "amount DOUBLE, event_ts TIMESTAMP")
```

### Schema Evolution Modes

What happens when new files arrive with new columns?

| Mode | Behaviour | When to use |
|------|-----------|-------------|
| `addNewColumns` *(default)* | Adds new columns to the schema; stream restarts automatically | Most Bronze ingestion |
| `rescue` | Unexpected columns are captured in a `_rescued_data` JSON string column; stream continues without restart | When you want continuity over schema completeness |
| `failOnNewColumns` | Stream fails if new columns arrive | Strict schema control |
| `none` | Ignores new columns silently | When new columns are expected and unwanted |

```python
.option("cloudFiles.schemaEvolutionMode", "rescue")   # or addNewColumns, failOnNewColumns, none
```

### The `_rescued_data` Column

When `schemaEvolutionMode = rescue`, any column in the source file that doesn't match the current schema is serialized into a JSON string in `_rescued_data`. This ensures no data is lost even when the schema doesn't match.

```json
// Source file has: {"order_id": "ORD001", "amount": 120.0, "new_field": "xyz"}
// Schema has: order_id, amount
// _rescued_data: '{"new_field": "xyz"}'
```

> **Exam tip:** `_rescued_data` appears automatically in `rescue` mode — you don't configure the column name. It stores values as a JSON string, not as structured columns.

## File Discovery Modes

Auto Loader has two modes for detecting new files:

### Directory Listing (default)

Auto Loader lists the source directory and compares file names against what's already been processed (tracked in the checkpoint). Simple to set up — no cloud configuration needed.

- Works well up to ~**10,000 files** per trigger
- Scales poorly for very large directories — listing millions of files is expensive

### File Notification

Cloud storage is configured to emit an event (S3 Event → SQS, ADLS → Event Grid) whenever a new file arrives. Auto Loader subscribes to these events instead of listing the directory.

```python
.option("cloudFiles.useNotifications", "true")     # enable file notification mode
.option("cloudFiles.region",           "eu-west-1") # cloud region
```

- Scales to **millions of files** — event-driven, no directory listing
- Requires cloud-side setup (SQS queue, permissions)
- Lower latency — file is processed as soon as the event arrives

| | Directory Listing | File Notification |
|---|---|---|
| Setup | Zero config | Cloud queue + permissions |
| Scale | ~10K files | Millions of files |
| Latency | Next trigger cycle | Near real-time |
| Cost | List API calls | Event/queue costs |

## Checkpointing & Trigger Modes

### Checkpoint

The checkpoint stores:
- **Which files have been processed** (so they are never re-read)
- **Stream offset** — progress within the current batch
- **Inferred schema** (if `schemaLocation` points to the same path)

The checkpoint is **required** for exactly-once guarantees. Always store it on durable cloud storage, not on the driver's local disk.

```python
.option("checkpointLocation", "s3://my-bucket/checkpoints/events")
```

> Deleting the checkpoint forces Auto Loader to reprocess all files from the beginning — useful for full reprocessing, but can create duplicates if the target table already has data.

### Trigger Modes

| Trigger | Behaviour | Typical use |
|---------|-----------|-------------|
| `trigger(availableNow=True)` | Processes all files that have arrived since last run, then stops | Scheduled batch-style runs (preferred over `once`) |
| `trigger(processingTime="5 minutes")` | Runs a micro-batch every N minutes, runs continuously | Continuous near-real-time ingestion |
| `trigger(once=True)` | Processes one micro-batch then stops *(deprecated)* | Use `availableNow` instead |
| No trigger (default) | Continuous processing — micro-batch as fast as possible | Streaming with minimum latency |

> **Exam tip:** `availableNow=True` is the modern replacement for `once=True`. It processes **all available** files in one or more micro-batches (not just one), then stops — more efficient than `once`.

## Adding Source Metadata

Auto Loader exposes the `_metadata` column for each file row — the same metadata column available in direct file queries:

```python
from pyspark.sql.functions import col

stream = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "json")
         .option("cloudFiles.schemaLocation", "/ckpt/schema")
         .load("/landing/events/")
         .select(
             "*",
             col("_metadata.file_path").alias("source_file"),
             col("_metadata.file_modification_time").alias("file_ts")
         )
)
```

Including `source_file` in the Bronze table is a best practice — it enables auditing, debugging, and reprocessing individual source files.

## Auto Loader vs COPY INTO

Both Auto Loader and `COPY INTO` ingest new files incrementally and idempotently. The choice depends on volume, latency, and operational simplicity:

| | Auto Loader | COPY INTO |
|---|---|---|
| **API** | Structured Streaming (`readStream`) | SQL command |
| **Trigger** | Continuous or `availableNow` scheduled | Manual / scheduled job |
| **File scale** | Millions of files (file notification) | Thousands of files |
| **Schema inference** | Yes — with evolution support | Yes — basic |
| **Idempotent** | Yes (checkpoint) | Yes (Delta log tracking) |
| **Transform inline** | Full Spark SQL + Python | SELECT inside FROM clause |
| **Complexity** | Higher (checkpoint mgmt) | Lower (pure SQL) |
| **Best for** | High-volume continuous ingestion | Low-to-medium batch ingestion |

> **Decision rule:** If you expect more than ~10K files per run or need near-real-time latency, use Auto Loader. Otherwise, `COPY INTO` is simpler and sufficient.

## Hands-On: Auto Loader Pipeline

In [ ]:
import json
from datetime import datetime, timedelta

# Paths — all under /tmp/auto_loader_demo/
SOURCE_PATH  = "dbfs:/tmp/auto_loader_demo/source/"
BRONZE_PATH  = "dbfs:/tmp/auto_loader_demo/bronze/"
SCHEMA_PATH  = "dbfs:/tmp/auto_loader_demo/schema/"
CKPT_PATH    = "dbfs:/tmp/auto_loader_demo/checkpoint/"

# Clean up any previous run
dbutils.fs.rm("dbfs:/tmp/auto_loader_demo/", recurse=True)
dbutils.fs.mkdirs(SOURCE_PATH)

# Write batch 1 — two files
batch1_file1 = [{"order_id": "ORD001", "amount": 120.5,  "region": "UK"},
                {"order_id": "ORD002", "amount": 340.0,  "region": "US"}]
batch1_file2 = [{"order_id": "ORD003", "amount":  89.99, "region": "UK"},
                {"order_id": "ORD004", "amount": 210.0,  "region": "DE"}]

dbutils.fs.put(SOURCE_PATH + "file_001.json",
               "\n".join(json.dumps(r) for r in batch1_file1), overwrite=True)
dbutils.fs.put(SOURCE_PATH + "file_002.json",
               "\n".join(json.dumps(r) for r in batch1_file2), overwrite=True)

print("Source files written:")
display(dbutils.fs.ls(SOURCE_PATH))

In [ ]:
from pyspark.sql.functions import col, current_timestamp

# Define the Auto Loader stream — adds metadata columns for auditing
def run_auto_loader():
    return (
        spark.readStream
             .format("cloudFiles")
             .option("cloudFiles.format",          "json")
             .option("cloudFiles.schemaLocation",  SCHEMA_PATH)
             .option("cloudFiles.inferColumnTypes", "true")
             .load(SOURCE_PATH)
             .select(
                 "*",
                 col("_metadata.file_name").alias("source_file"),
                 col("_metadata.file_modification_time").alias("file_ts"),
                 current_timestamp().alias("ingested_at")
             )
             .writeStream
             .format("delta")
             .outputMode("append")
             .option("checkpointLocation", CKPT_PATH)
             .trigger(availableNow=True)  # process all new files then stop
             .start(BRONZE_PATH)
    )

# Run 1 — processes file_001.json and file_002.json
query = run_auto_loader()
query.awaitTermination()
print("Run 1 complete")

In [ ]:
# Inspect what was written
bronze_df = spark.read.format("delta").load(BRONZE_PATH)
print(f"Bronze rows after run 1: {bronze_df.count()}")
bronze_df.show(truncate=False)

In [ ]:
# Simulate new files arriving
batch2 = [{"order_id": "ORD005", "amount": 500.0, "region": "FR"},
           {"order_id": "ORD006", "amount":  75.0, "region": "UK"}]

dbutils.fs.put(SOURCE_PATH + "file_003.json",
               "\n".join(json.dumps(r) for r in batch2), overwrite=True)

# Run 2 — only processes file_003.json (files 001 and 002 are in checkpoint)
query = run_auto_loader()
query.awaitTermination()
print("Run 2 complete")

bronze_df = spark.read.format("delta").load(BRONZE_PATH)
print(f"Bronze rows after run 2: {bronze_df.count()}")
bronze_df.orderBy("order_id").show(truncate=False)

In [ ]:
# Idempotency check — run again with no new files: row count unchanged
query = run_auto_loader()
query.awaitTermination()

final_count = spark.read.format("delta").load(BRONZE_PATH).count()
print(f"Row count after re-run with no new files: {final_count}  (should still be 6)")

In [ ]:
# Inspect schema evolution — checkpoint stores the inferred schema
print("Inferred schema stored by Auto Loader:")
bronze_df.printSchema()

In [ ]:
# Cleanup
dbutils.fs.rm("dbfs:/tmp/auto_loader_demo/", recurse=True)

## Production Auto Loader Pattern

A complete, production-ready Bronze ingestion pipeline:

```python
from pyspark.sql.functions import col, current_timestamp

(
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format",           "json")
         .option("cloudFiles.schemaLocation",   "s3://bucket/checkpoints/orders/schema")
         .option("cloudFiles.inferColumnTypes",  "true")
         .option("cloudFiles.schemaEvolutionMode", "rescue")   # never fail on new columns
         .option("cloudFiles.useNotifications",  "true")       # scale to millions of files
         .load("s3://bucket/landing/orders/")
         .select(
             "*",
             col("_metadata.file_path").alias("_source_file"),
             col("_metadata.file_modification_time").alias("_file_ts"),
             current_timestamp().alias("_ingested_at")
         )
         .writeStream
         .format("delta")
         .outputMode("append")
         .option("checkpointLocation", "s3://bucket/checkpoints/orders/stream")
         .option("mergeSchema",        "true")    # allow schema widening on write
         .trigger(availableNow=True)              # run as a scheduled job
         .toTable("bronze.orders")               # write to registered Delta table
)
```

Key production choices made here:
- `rescue` mode — never fail due to schema changes; capture unknowns in `_rescued_data`
- `useNotifications` — scale to millions of files
- Metadata columns prefixed with `_` — visually separate from source fields
- `toTable()` instead of `start(path)` — writes to a registered Delta table with governance
- `availableNow` — run as a Databricks Job on a schedule rather than as a continuous stream

## Summary

- **Auto Loader** incrementally ingests new files from cloud storage using the `cloudFiles` Structured Streaming source. It tracks processed files via a checkpoint — never re-reading files already ingested.
- Two required options: `cloudFiles.format` (source file format) and `cloudFiles.schemaLocation` (where to persist the inferred schema).
- `cloudFiles.inferColumnTypes=true` infers actual types; without it, all columns are strings.
- Four **schema evolution modes**: `addNewColumns` (default — adds columns, restarts stream), `rescue` (captures unknown columns in `_rescued_data` JSON string), `failOnNewColumns` (strict), `none` (ignores new columns).
- Two **file discovery modes**: directory listing (default, up to ~10K files) and file notification (event-driven via SQS/Event Grid, scales to millions).
- **Checkpoint** stores file tracking and stream offset. Deleting it forces a full reprocess from the beginning.
- **Trigger modes**: `availableNow=True` (process all pending files, then stop — preferred for scheduled jobs), `processingTime` (continuous micro-batch), `once` (deprecated).
- **`_metadata`** column exposes `file_path`, `file_name`, `file_modification_time` per row — include these in Bronze for auditability.
- Use **Auto Loader** for high-volume or continuous ingestion; use **`COPY INTO`** for simpler batch ingestion of thousands of files.

Next: `10-change-data-capture-merge.ipynb` — processing CDC feeds with `MERGE INTO` and SCD patterns.